# Audit DB Basics: How to Write Event Data Reliably

## Recording Agent Activity

**Difficulty: Beginner | ~35 min | Requires Supabase setup (see Prerequisites)**

*Lab 1 of 5 in the Audit DB Labs module.*

An AI harness does real work — it calls tools, queries databases, spends money — but none of that is visible while it happens. If you can't answer *"what exactly did the agent do, and when?"*, the harness is a black box you can't debug, audit, or trust.

In this lab you build the smallest useful **audit log** for an agent, on a real cloud Postgres database (Supabase), step by step:

1. Connect to Postgres using credentials from your `.env` file (never typed into code).
2. Create two tables: a `run` header table and an `event` table.
3. Record one run and its events inside a single **transaction**.
4. Close the run with its one-and-only lifecycle update.
5. Read the history back with `SELECT` ... `WHERE` ... `ORDER BY`.
6. Fix a mislogged event with a **correction** (never an edit).
7. **Redact** a captured secret — visibly, with its own log entry.
8. Recap what the finished log proves.

Everything runs against your real database, so every output you see was truly written.

In [1]:
!pip install python-dotenv==1.2.3 psycopg2-binary==2.9.12

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.0.1
    Uninstalling python-dotenv-1.0.1:
      Successfully uninstalled python-dotenv-1.0.1



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


One pinned line installs everything the lab needs, matching Section 9 exactly.

### Step 1 — Connect to Postgres

In [2]:
import os
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
import psycopg2

# Walk upward until we find the module's .env so credentials stay out of this notebook.
env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")
assert database_url, "DATABASE_URL not found - complete README Section 7 setup first."

connection = psycopg2.connect(database_url, connect_timeout=10)
cursor = connection.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])

PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


We need four things here, and each earns its line:

- `psycopg2` is the standard Python driver for Postgres — it opens the network connection and ships SQL to it.
- Your credentials live in a `.env` file at the module level (set up in README Section 7). Loading them at runtime keeps secrets out of the notebook file, out of screenshots, and out of git.
- The small search expression walks *upward* from this notebook's folder until it finds `.env`, so the same code works no matter which lab folder it runs from.
- Finally we ask Postgres for its version string — proof the connection really succeeded before anything else happens.

### Step 2 — Create the Two Tables

In [3]:
create_audit_tables = """
    CREATE TABLE IF NOT EXISTS run (
        run_id      SERIAL PRIMARY KEY,
        agent_name  TEXT,
        started_at  TIMESTAMPTZ DEFAULT now(),
        ended_at    TIMESTAMPTZ,
        status      TEXT,
        total_cost  NUMERIC(10, 6));
    CREATE TABLE IF NOT EXISTS event (
        event_id   SERIAL PRIMARY KEY,
        run_id     INTEGER,
        event_type TEXT,
        payload    TEXT,
        created_at TIMESTAMPTZ DEFAULT now());
"""
cursor.execute(create_audit_tables)
connection.commit()
print("Tables ready: run, event")

Tables ready: run, event


An audit log needs only two shapes of row for now:

- A `run` row is the **header**: one per agent invocation, holding who ran, when it started and ended, how it ended, and what it cost.
- An `event` row is one thing that happened *during* a run. Its plain `run_id` column says which run it belongs to.

`IF NOT EXISTS` makes this safe to re-run — tables are created once, then reused. Two details worth noticing: `SERIAL PRIMARY KEY` auto-assigns each row's permanent id (the bare minimum wiring this lab needs to point at individual rows — foreign keys, `CHECK`/`NOT NULL` rules, and indexes come in Lab 2), and `DEFAULT now()` stamps the time columns automatically, because timestamps are an audit log's backbone.

### Step 3 — Record a Run and Its Events in One Transaction

In [4]:
cursor.execute(
    "INSERT INTO run (agent_name, status) VALUES (%s, %s) RETURNING run_id",
    ("support-agent", "running"),
)
current_run_id = cursor.fetchone()[0]

events_to_log = [
    ("user_message", "User asked: how many orders shipped yesterday?"),
    ("db_query", "Counted shipped orders in the orders table."),
    ("model_answer", "Answered: 1284 orders shipped yesterday."),
]
for event_type, payload_text in events_to_log:
    cursor.execute(
        "INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s)",
        (current_run_id, event_type, payload_text),
    )

# ONE commit saves the header and every event together; any error before it means NOTHING is saved.
connection.commit()
print(f"Run {current_run_id} committed with {len(events_to_log)} events in one transaction.")

Run 8 committed with 3 events in one transaction.


A **transaction** groups several writes so they succeed or fail *together* — never half-applied (README Section 4.1). With `psycopg2` a transaction is already open; nothing touches the database until you call `commit()`.

The order matters: insert the `run` header first and use `RETURNING run_id` to catch the id Postgres generated for it. Every event after that carries that id, so the whole history hangs together. One `commit()` at the end saves the header and all three events as one unit.

### Step 4 — Close the Run: The One Allowed Update

In [5]:
# The ONE update a run row ever gets: closing it out once the run finishes (README Section 3).
cursor.execute(
    "UPDATE run SET ended_at = now(), status = 'completed', total_cost = %s WHERE run_id = %s",
    (0.0042, current_run_id),
)
connection.commit()
print(f"Run {current_run_id} closed: status='completed', total_cost=$0.0042")

Run 8 closed: status='completed', total_cost=$0.0042


Everything else in an audit log is append-only, but the `run` row gets exactly one exception (README Section 3): when the run finishes, we fill in `ended_at`, `status`, and `total_cost` — once. That isn't rewriting history; the run genuinely wasn't finished when the row was created, so there was nothing yet to misrepresent.

This is the **only** time this notebook updates the `run` row. After this cell the header is frozen. Every later step appends new `event` rows — or redacts one field under the strict rule in Step 7. No other edits, ever.

### Step 5 — Read the History Back

In [6]:
cursor.execute(
    """SELECT event_id, event_type, payload, created_at
       FROM event
       WHERE run_id = %s
       ORDER BY created_at, event_id""",
    (current_run_id,),
)
for row in cursor.fetchall():
    print(row)

(11, 'user_message', 'User asked: how many orders shipped yesterday?', datetime.datetime(2026, 8, 24, 18, 50, 35, 74599, tzinfo=datetime.timezone.utc))
(12, 'db_query', 'Counted shipped orders in the orders table.', datetime.datetime(2026, 8, 24, 18, 50, 35, 74599, tzinfo=datetime.timezone.utc))
(13, 'model_answer', 'Answered: 1284 orders shipped yesterday.', datetime.datetime(2026, 8, 24, 18, 50, 35, 74599, tzinfo=datetime.timezone.utc))


This is the query an auditor actually runs: *what happened during this run, in order?* `WHERE` picks out just our run's rows, and `ORDER BY created_at` replays them in time order.

The `, event_id` tiebreaker isn't decoration: events committed in the same transaction share an identical timestamp (`now()` returns the transaction's start time), so without it Postgres would return ties in arbitrary order.

### Step 6 — Corrections Instead of Edits

In [7]:
cursor.execute(
    "INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s) RETURNING event_id",
    (current_run_id, "db_query", "Order-count query scanned 1500 rows."),
)
wrong_event_id = cursor.fetchone()[0]

# The correction is a NEW event naming what it fixes; the wrong row stays untouched.
cursor.execute(
    "INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s) RETURNING event_id",
    (current_run_id, "correction",
     f"Correction: event {wrong_event_id} reported 1500 rows scanned; true count was 15000."),
)
correcting_event_id = cursor.fetchone()[0]
connection.commit()

cursor.execute("SELECT event_id, event_type, payload FROM event WHERE event_id IN (%s, %s)",
               (wrong_event_id, correcting_event_id))
for row in cursor.fetchall():
    print(row)

(14, 'db_query', 'Order-count query scanned 1500 rows.')
(15, 'correction', 'Correction: event 14 reported 1500 rows scanned; true count was 15000.')


Suppose the harness logged a wrong number. In a normal database you'd `UPDATE` the row and the mistake would vanish. Here that's forbidden (README Section 3.1): a history you can quietly rewrite is worthless to an auditor.

Instead we insert a **new** `correction` event that names the row it fixes and states the true value. Both rows stay visible forever — the mistake remains part of the history, which is exactly the point.

### Step 7 — Redaction Instead of Deletion

In [8]:
cursor.execute(
    "INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s) RETURNING event_id",
    (current_run_id, "auth_check", "Login verified with api_key=sk-demo-secret-42."),
)
secret_event_id = cursor.fetchone()[0]

cursor.execute(
    "UPDATE event SET payload = REPLACE(payload, %s, %s) WHERE event_id = %s",
    ("sk-demo-secret-42", "[REDACTED]", secret_event_id),
)
# Part 2 logs the redaction: which event, which field, when (UTC).
cursor.execute(
    "INSERT INTO event (run_id, event_type, payload) VALUES (%s, %s, %s) RETURNING event_id",
    (current_run_id, "redaction",
     f"Redacted field 'payload' of event {secret_event_id} at {datetime.now(timezone.utc)}"),
)
redaction_log_id = cursor.fetchone()[0]
connection.commit()

cursor.execute("SELECT event_id, event_type, payload FROM event WHERE event_id IN (%s, %s)",
               (secret_event_id, redaction_log_id))
for row in cursor.fetchall():
    print(row)

(16, 'auth_check', 'Login verified with api_key=[REDACTED].')
(17, 'redaction', "Redacted field 'payload' of event 16 at 2026-08-24 18:50:40.821485+00:00")


During login, the harness captured a secret into an event's payload (README Section 3.2). Sensitive data is the one deliberate exception to "event rows never change" — and even the exception has a rule:

1. Replace **just the secret inside that field** — the row itself stays, with its timestamp and existence intact.
2. Insert a **new event logging the redaction**: which event, which field, when.

A silent overwrite would hide the fact that something was hidden. Logging the redaction keeps even the cleanup on the record. (The key below is fake demo data — but the pattern is exactly what production systems do.)

### Step 8 — Recap

In [9]:
# Release the connection - good hygiene whenever a notebook finishes talking to a database.
cursor.close()
connection.close()

print(f"Run {current_run_id}: opened, logged, closed, corrected, redacted - history intact.")
print("The mislogged row AND the redaction note are both still visible; nothing was silently rewritten.")

Run 8: opened, logged, closed, corrected, redacted - history intact.
The mislogged row AND the redaction note are both still visible; nothing was silently rewritten.


Close the connection politely, then print what this run's audit log now proves. Notice the shape of the story: a run opened, events appended inside a transaction, the header closed exactly once, a mistake corrected by appending, a secret redacted on the record — and not one row silently rewritten or deleted along the way. That discipline is the foundation every later lab in this module builds on.